# Position bias [Step 07.02 - Measuring the flip rate]

> **MLCourse - Agentic AI - Agent Patterns**

Pairwise judging - "which of these two is better?" - is easier for a model than
assigning absolute scores, so it is what most evaluation harnesses use.

It has one large, well-documented defect: **the judge's answer depends on which
answer you put first.** Not "sometimes, slightly". Measurably, and often enough to
reverse a leaderboard.

The test is simple and you should run it on every judge you deploy:

```
ask:  (question, A=x, B=y)  -> verdict 1
ask:  (question, A=y, B=x)  -> verdict 2      <-- same content, swapped order

CONSISTENT  if the two verdicts name the same ANSWER
FLIPPED     if they name the same POSITION
```

The **flip rate** is the fraction of pairs that flip. A perfect judge flips 0% of
the time. A judge that flips 100% of the time is reading position and ignoring
content entirely.

### Key takeaways

- Always evaluate **both orders** and count agreement. Costs 2x and is not optional.
- A flip is not a coin-flip error - it is a *systematic* preference for one slot,
  so it does not average out over more comparisons.
- The standard mitigations are **swap-and-average** and **ties**: if the two orders
  disagree, record a tie rather than picking one.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 2.0
print("PACE =", PACE)

PACE = 2.0


### The evaluation pairs


In [ ]:
# Five questions, each with a GOOD answer and a WORSE answer. "Worse" is worse for
# a stated, checkable reason - not just shorter or blander - so we know the right
# verdict without asking anyone.
#
# Building the pairs by hand like this is the only way to measure a judge: you need
# ground truth ABOUT THE JUDGE, which means you must know the answer already.

PAIRS = [
    dict(
        id="photosynthesis",
        q="In one or two sentences, what does photosynthesis produce?",
        good="Photosynthesis produces glucose and oxygen, using carbon dioxide, water and light energy.",
        bad="Photosynthesis produces carbon dioxide and water, which the plant then releases into the air.",
        why_bad="reverses the reactants and the products - factually wrong",
    ),
    dict(
        id="http_status",
        q="What does HTTP status code 404 mean?",
        good="404 means the server understood the request but could not find the requested resource.",
        bad="404 means the server is temporarily overloaded and the client should retry later.",
        why_bad="describes 503, not 404",
    ),
    dict(
        id="python_list",
        q="What is the time complexity of appending to a Python list?",
        good="Amortised O(1): appends are constant time on average, with occasional O(n) reallocations.",
        bad="O(n), because the list has to be copied every time an element is added.",
        why_bad="wrong complexity - ignores the amortised growth strategy",
    ),
    dict(
        id="vaccine",
        q="Briefly, how do mRNA vaccines work?",
        good=("They deliver mRNA instructing your cells to make a harmless viral protein, "
              "which the immune system then learns to recognise."),
        bad=("They inject a weakened live virus that reproduces slowly so the immune system "
             "can practise fighting it."),
        why_bad="describes a live attenuated vaccine, not an mRNA vaccine",
    ),
    dict(
        id="git_rebase",
        q="What does `git rebase` do, in one sentence?",
        good="It replays your commits on top of another base commit, rewriting their history.",
        bad="It merges two branches together and creates a merge commit recording both parents.",
        why_bad="describes merge, which is the thing rebase is contrasted with",
    ),
]

print("%d pairs" % len(PAIRS))
for p in PAIRS:
    print("  %-16s bad answer: %s" % (p["id"], p["why_bad"]))


### A pairwise judge


In [ ]:
# Reply format is fixed so a REGEX reads the verdict. Never parse a judge's verdict
# with another LLM: you would then need to evaluate that one too.

import re

judge_llm = make_llm(temperature=0.0, max_tokens=200)

PAIRWISE_SYSTEM = (
    "You are an impartial evaluator. You will see a question and two candidate "
    "answers, A and B. Decide which answer is more FACTUALLY CORRECT. "
    "Ignore length, tone, formatting and confidence. "
    "Reply with one short sentence of justification, then a final line of exactly "
    "the form 'VERDICT: A' or 'VERDICT: B'.")

VERDICT_RE = re.compile(r"VERDICT:\s*([AB])")


def judge_pair(question, answer_a, answer_b):
    """Return ('A'|'B'|'?', raw_text)."""
    m = safe_invoke(judge_llm, [
        ("system", PAIRWISE_SYSTEM),
        ("user", "QUESTION:\n%s\n\nANSWER A:\n%s\n\nANSWER B:\n%s"
                 % (question, answer_a, answer_b))])
    v = VERDICT_RE.search(m.content)
    return (v.group(1) if v else "?"), m.content


### 1. Both orders, every pair

For each pair we run the judge twice. Because we constructed the pairs, we know
which answer *should* win, so we can measure two separate things:

- **accuracy** - did it pick the correct answer?
- **consistency** - did it pick the *same* answer both times?

These are different failures and they need separating. A judge can be consistently
wrong (a knowledge problem) or inconsistently right (a position problem).

In [5]:
rows = []
for p in PAIRS:
    # order 1: good is A
    v1, t1 = judge_pair(p["q"], p["good"], p["bad"])
    # order 2: good is B
    v2, t2 = judge_pair(p["q"], p["bad"], p["good"])

    picked_1 = "good" if v1 == "A" else ("bad" if v1 == "B" else "?")
    picked_2 = "good" if v2 == "B" else ("bad" if v2 == "A" else "?")
    consistent = picked_1 == picked_2 and picked_1 != "?"
    rows.append(dict(id=p["id"], v1=v1, v2=v2, picked_1=picked_1, picked_2=picked_2,
                     consistent=consistent))
    print("%-16s order1 VERDICT %s -> %-5s | order2 VERDICT %s -> %-5s | %s"
          % (p["id"], v1, picked_1, v2, picked_2,
             "consistent" if consistent else "FLIPPED"))

photosynthesis   order1 VERDICT A -> good  | order2 VERDICT B -> good  | consistent


http_status      order1 VERDICT A -> good  | order2 VERDICT B -> good  | consistent


python_list      order1 VERDICT A -> good  | order2 VERDICT B -> good  | consistent


vaccine          order1 VERDICT A -> good  | order2 VERDICT B -> good  | consistent


git_rebase       order1 VERDICT A -> good  | order2 VERDICT B -> good  | consistent


In [6]:
n = len(rows)
flips = [r for r in rows if not r["consistent"]]
correct_1 = sum(1 for r in rows if r["picked_1"] == "good")
correct_2 = sum(1 for r in rows if r["picked_2"] == "good")
always_A = sum(1 for r in rows if r["v1"] == "A" and r["v2"] == "A")
always_B = sum(1 for r in rows if r["v1"] == "B" and r["v2"] == "B")

print("=" * 62)
print("POSITION BIAS MEASUREMENT (n=%d pairs, %d judge calls)" % (n, 2 * n))
print("-" * 62)
print("FLIP RATE                          : %d/%d = %.0f%%" % (len(flips), n, 100 * len(flips) / n))
print("accuracy with good answer in slot A: %d/%d = %.0f%%" % (correct_1, n, 100 * correct_1 / n))
print("accuracy with good answer in slot B: %d/%d = %.0f%%" % (correct_2, n, 100 * correct_2 / n))
print("picked slot A both times           : %d" % always_A)
print("picked slot B both times           : %d" % always_B)
print("=" * 62)
if len(flips) == 0:
    print("No flips on this set. That is a good sign and NOT a clean bill of health:")
    print("5 pairs cannot detect a flip rate below about 20%. The published")
    print("literature reports double-digit flip rates on harder, closer pairs -")
    print("ours are deliberately easy (one answer is plainly wrong), which is")
    print("exactly the regime where judges do best.")
else:
    print("Flipped pairs:")
    for r in flips:
        print("  %-16s slot-A pick both times? v1=%s v2=%s" % (r["id"], r["v1"], r["v2"]))

POSITION BIAS MEASUREMENT (n=5 pairs, 10 judge calls)
--------------------------------------------------------------
FLIP RATE                          : 0/5 = 0%
accuracy with good answer in slot A: 5/5 = 100%
accuracy with good answer in slot B: 5/5 = 100%
picked slot A both times           : 0
picked slot B both times           : 0
No flips on this set. That is a good sign and NOT a clean bill of health:
5 pairs cannot detect a flip rate below about 20%. The published
literature reports double-digit flip rates on harder, closer pairs -
ours are deliberately easy (one answer is plainly wrong), which is
exactly the regime where judges do best.


### Why our pairs are the easy case, and what to do about it

Our bad answers are **plainly** wrong - they describe a different concept
altogether. Position bias shows up most strongly when the two answers are **close
in quality**, because then position is the only signal left. That is also the regime
your real evaluations live in: you are usually comparing v1 of your system against
v2, and they are similar by construction.

So let's construct the hard case: two answers that are both correct, differing only
in length and polish. There is no right answer here - which is the point. A judge
with no position bias should be near 50/50 across the two orders, and above all it
should be **consistent** with itself.

In [7]:
CLOSE = [
    dict(id="tcp_udp",
         q="What is the main difference between TCP and UDP?",
         x="TCP is connection-oriented and guarantees ordered, reliable delivery; UDP is connectionless and does neither.",
         y="TCP sets up a connection and makes sure packets arrive in order without loss, whereas UDP just fires packets off with no such guarantees."),
    dict(id="index",
         q="Why does a database index speed up queries?",
         x="An index is a sorted structure that lets the engine find matching rows without scanning the whole table.",
         y="Because the database can look values up in a pre-sorted structure instead of reading every row, turning a full scan into a much cheaper lookup."),
    dict(id="cache",
         q="What is a cache hit ratio?",
         x="The fraction of requests served from the cache rather than the underlying store.",
         y="It is the proportion of lookups that the cache was able to answer itself, as opposed to having to go to the slower backing store."),
]

close_rows = []
for c in CLOSE:
    v1, _ = judge_pair(c["q"], c["x"], c["y"])
    v2, _ = judge_pair(c["q"], c["y"], c["x"])
    picked_1 = "x" if v1 == "A" else "y"
    picked_2 = "y" if v2 == "A" else "x"      # in order 2, slot A holds y
    consistent = picked_1 == picked_2
    close_rows.append((c["id"], v1, v2, picked_1, picked_2, consistent))
    print("%-10s order1 -> %s (%s) | order2 -> %s (%s) | %s"
          % (c["id"], v1, picked_1, v2, picked_2,
             "consistent" if consistent else "FLIPPED"))

tcp_udp    order1 -> A (x) | order2 -> B (x) | consistent


index      order1 -> ? (y) | order2 -> A (y) | consistent


cache      order1 -> A (x) | order2 -> A (y) | FLIPPED


In [8]:
cn = len(close_rows)
cflips = sum(1 for r in close_rows if not r[5])
slotA = sum(1 for r in close_rows if r[1] == "A") + sum(1 for r in close_rows if r[2] == "A")

print("=" * 62)
print("CLOSE-QUALITY PAIRS (n=%d, %d judge calls)" % (cn, 2 * cn))
print("-" * 62)
print("FLIP RATE           : %d/%d = %.0f%%" % (cflips, cn, 100 * cflips / cn))
print("slot A chosen       : %d/%d of all verdicts = %.0f%%" % (slotA, 2 * cn, 100 * slotA / (2 * cn)))
print("=" * 62)
print()
print("Compare against the easy pairs above. The flip rate on close pairs is the")
print("number that predicts what your real evaluation harness will do, because")
print("your real comparisons are close by construction.")

CLOSE-QUALITY PAIRS (n=3, 6 judge calls)
--------------------------------------------------------------
FLIP RATE           : 1/3 = 33%
slot A chosen       : 4/6 of all verdicts = 67%

Compare against the easy pairs above. The flip rate on close pairs is the
number that predicts what your real evaluation harness will do, because
your real comparisons are close by construction.


### 2. The mitigations

Three, in increasing order of cost and reliability.

**(a) Swap and require agreement.** Run both orders. If they disagree, record a
**tie** rather than picking a winner. This is what Chatbot Arena-style harnesses do
and it is the minimum acceptable practice.

**(b) Swap and average.** For pointwise-style scores rather than verdicts, average
the two orders. Removes the bias in expectation, not per-comparison.

**(c) Randomise position per comparison.** Cheaper than (a) - one call, not two -
and it converts a *systematic* bias into *noise*. Noise is much less dangerous: it
widens your error bars instead of shifting your conclusion. Use this when you cannot
afford 2x.

Here is (a), implemented.

In [9]:
def judge_pair_symmetric(question, answer_x, answer_y):
    """Judge both orders. Returns 'x', 'y' or 'tie'.

    A tie means the judge contradicted itself - which is real information, not a
    failure. Recording it as a tie is honest; picking one order's answer is not.
    """
    v1, _ = judge_pair(question, answer_x, answer_y)
    v2, _ = judge_pair(question, answer_y, answer_x)
    pick_1 = "x" if v1 == "A" else "y"
    pick_2 = "y" if v2 == "A" else "x"
    return pick_1 if pick_1 == pick_2 else "tie"


print("symmetric judging on the close pairs (2 calls each):")
for c in CLOSE:
    print("  %-10s -> %s" % (c["id"], judge_pair_symmetric(c["q"], c["x"], c["y"])))

symmetric judging on the close pairs (2 calls each):


  tcp_udp    -> tie


  index      -> y


  cache      -> tie


### Pitfalls

- **Reporting a win rate from single-order comparisons.** If you did not swap, your
  win rate contains your judge's slot preference and you cannot separate the two.
- **Treating ties as losses (or wins).** Report them as their own category. A
  harness with 40% ties is telling you the two systems are indistinguishable to
  this judge, which is a finding.
- **Assuming the bias favours slot A.** It varies by model and by prompt. Measure
  the direction, do not assume it.
- **Re-measuring only once.** The flip rate changes when you change the rubric, the
  model version, or the answer length distribution. It is a property of the whole
  setup, not of "LLM judges".

### Next

Notebook 03 covers the two biases that survive swapping: the judge preferring
**longer** answers, and preferring **its own**.